In [ ]:
# CELL 1: Configuration - recursive Oracle documentation roots by platform
# Each TOC URL is crawled recursively within its own Oracle documentation guide directory.
PLATFORM_URLS = {
    "Exadata On-Prem": [
        "https://docs.oracle.com/en/engineered-systems/exadata-database-machine/dbmin/toc.htm",
        "https://docs.oracle.com/en/engineered-systems/exadata-database-machine/sagug/toc.htm",
        "https://docs.oracle.com/en/engineered-systems/exadata-database-machine/dbmso/toc.htm",
        "https://docs.oracle.com/en/engineered-systems/exadata-database-machine/dbmsq/toc.htm",
        "https://docs.oracle.com/en/engineered-systems/exadata-database-machine/dbmmn/toc.htm",
    ],
    "Exascale": [
        "https://docs.oracle.com/en/engineered-systems/exadata-database-machine/exscl/toc.htm",
    ],
    "ExaCC": [
        "https://docs.oracle.com/en/engineered-systems/exadata-cloud-at-customer/ecccm/toc.htm",
    ],
    "ExaCS": [
        "https://docs.oracle.com/en/engineered-systems/exadata-cloud-service/ecscm/toc.htm",
    ],
}

MAX_DEPTH = 8
MAX_PAGES = 3000
REQUEST_TIMEOUT = 30
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200


In [ ]:
# CELL 2: Imports and recursive URL crawler
import os
import re
import shutil
from io import BytesIO
from collections import deque
from urllib.parse import urljoin, urlparse, urldefrag, parse_qsl, urlencode

import requests
from bs4 import BeautifulSoup
from pypdf import PdfReader
from langchain_core.documents import Document

def normalize_url(url):
    url, _ = urldefrag(url)
    parsed = urlparse(url)
    if parsed.scheme not in {"http", "https"} or not parsed.netloc:
        return None
    # Drop tracking/query parameters while retaining meaningful query URLs.
    kept_query = [(k, v) for k, v in parse_qsl(parsed.query)
                  if not k.lower().startswith(("utm_", "source", "ref"))]
    return parsed._replace(query=urlencode(kept_query), fragment="").geturl()

def scope_prefix(root_url):
    parsed = urlparse(root_url)
    path = parsed.path or "/"
    if not path.endswith("/"):
        path = path.rsplit("/", 1)[0] + "/"
    return (parsed.scheme, parsed.netloc, path)

def in_scope(url, root_scope):
    parsed = urlparse(url)
    scheme, netloc, prefix = root_scope
    return parsed.scheme == scheme and parsed.netloc == netloc and parsed.path.startswith(prefix)

def extract_html(html):
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript", "svg"]):
        tag.decompose()
    text = soup.get_text("\n", strip=True)
    text = re.sub(r"\n{3,}", "\n\n", text)
    links = [urljoin]  # placeholder removed below; keeps parser logic easy to read
    return text, soup

def crawl_platform(root_url, platform_name, max_depth=MAX_DEPTH, max_pages=MAX_PAGES):
    root_url = normalize_url(root_url)
    if not root_url:
        raise ValueError(f"Invalid root URL: {root_url}")

    root_scope = scope_prefix(root_url)
    queue = deque([(root_url, 0)])
    queued = {root_url}
    visited = set()
    documents = []
    session = requests.Session()
    session.headers.update({"User-Agent": "Oracle-Exadata-RAG-Doc-Crawler/1.0"})

    while queue and len(visited) < max_pages:
        url, depth = queue.popleft()
        if url in visited:
            continue
        visited.add(url)

        try:
            response = session.get(url, timeout=REQUEST_TIMEOUT, allow_redirects=True)
            response.raise_for_status()
        except requests.RequestException as exc:
            print(f"WARN: {url} -> {exc}")
            continue

        final_url = normalize_url(response.url) or url
        content_type = response.headers.get("content-type", "").lower()
        is_pdf = "application/pdf" in content_type or final_url.lower().endswith(".pdf")

        if is_pdf:
            try:
                reader = PdfReader(BytesIO(response.content))
                for page_num, page in enumerate(reader.pages):
                    text = (page.extract_text() or "").strip()
                    if text:
                        documents.append(Document(
                            page_content=text,
                            metadata={
                                "source": final_url,
                                "platform": platform_name,
                                "page": page_num,
                                "content_type": "pdf",
                            },
                        ))
            except Exception as exc:
                print(f"WARN: PDF parse failed for {final_url}: {exc}")
            continue

        if "text/html" not in content_type:
            continue

        soup = BeautifulSoup(response.text, "html.parser")
        for tag in soup(["script", "style", "noscript", "svg"]):
            tag.decompose()
        text = soup.get_text("\n", strip=True)
        text = re.sub(r"\n{3,}", "\n\n", text)
        if text:
            documents.append(Document(
                page_content=text,
                metadata={
                    "source": final_url,
                    "platform": platform_name,
                    "depth": depth,
                    "content_type": "html",
                },
            ))

        if depth >= max_depth:
            continue

        for anchor in soup.find_all("a", href=True):
            child = normalize_url(urljoin(final_url, anchor["href"]))
            if child and child not in visited and child not in queued and in_scope(child, root_scope):
                queued.add(child)
                queue.append((child, depth + 1))

    print(f"{platform_name}: visited={len(visited)}, queue_remaining={len(queue)}, documents={len(documents)}")
    return documents


In [ ]:
# CELL 3: Crawl every configured root recursively
all_docs = []
for platform_name, root_urls in PLATFORM_URLS.items():
    print(f"\n=== Crawling {platform_name} ===")
    for root_url in root_urls:
        print(f"Root: {root_url}")
        platform_docs = crawl_platform(root_url, platform_name)
        all_docs.extend(platform_docs)

print(f"\nTotal source documents collected: {len(all_docs)}")


In [ ]:
# CELL 4: Split documents while preserving URL/platform/page metadata
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

docs_chunks = text_splitter.split_documents(all_docs)
print(f"Total chunks generated: {len(docs_chunks)}")
print("Sample metadata:", docs_chunks[0].metadata if docs_chunks else "No documents")


In [ ]:
# CELL 5: Local HuggingFace embeddings
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print("Embedding model ready.")


In [ ]:
# CELL 6: Build one Chroma database per selectable platform
from langchain_chroma import Chroma

DB_FOLDERS = {
    "Exadata On-Prem": "./vectordb_exadata_onprem",
    "Exascale": "./vectordb_exascale",
    "ExaCC": "./vectordb_exacc",
    "ExaCS": "./vectordb_exacs",
}

for platform_name, folder in DB_FOLDERS.items():
    platform_chunks = [d for d in docs_chunks if d.metadata.get("platform") == platform_name]
    if not platform_chunks:
        print(f"Skipping {platform_name}: no documents collected")
        continue

    if os.path.exists(folder):
        shutil.rmtree(folder)

    Chroma.from_documents(
        documents=platform_chunks,
        embedding=embedding,
        persist_directory=folder,
        collection_name="exadata_docs",
    )
    print(f"Created {folder} with {len(platform_chunks)} chunks")

print("\nAll platform indexes are ready for the Streamlit app.")


In [ ]:
# CELL 7: Optional quick sanity check
for platform_name, folder in DB_FOLDERS.items():
    if os.path.exists(folder):
        test_db = Chroma(persist_directory=folder, embedding_function=embedding, collection_name="exadata_docs")
        print(platform_name, "->", test_db._collection.count(), "chunks")
